# Inference with Base E2ETune Model (No Adapter)

This notebook demonstrates how to load the base E2ETune model (`springhxm/E2ETune`) directly from Hugging Face and run inference **without** applying any fine-tuned LoRA adapters.

In [ ]:
!pip install -U transformers bitsandbytes accelerate huggingface_hub

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import os

# Configuration
BASE_MODEL_ID = "springhxm/E2ETune"

# 1. Load Base Model and Tokenizer
We load the base model in 4-bit precision to save GPU memory. Notice we skip the `peft` adapter loading here entirely.

In [ ]:
print("Configuring 4-bit quantization...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

print(f"Loading Base Model: {BASE_MODEL_ID}...")
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    use_cache=True,
    use_safetensors=True
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
# Switch tokenizer to LEFT padding for generation since we will anchor the end
tokenizer.padding_side = "left"

model.eval()
print("Base model ready for inference!")

# 2. JSON Inference Input
Dynamically load the workload and internal metrics from a JSON file, apply the token-saving deduplication for query plans, and format it into the Mistral Instruct format `[INST] ... [/INST] {` anchored prompt.

In [ ]:
import json
import re
from collections import defaultdict

# Replace this with the path to your JSON file
input_json_path = "/home/E2ETune-AI4DB/data/mysql/hetzner-4c-8t-32gb/job/job_1/collected_data.json"

db_name = "MYSQL" # Update to POSTGRESQL if using Postgres data
hw_specs = "hetzner-4c-8t-32gb"

with open(input_json_path, "r") as f:
    data = json.load(f)

# Parse Internal Metrics
metrics_str = ", ".join([f"{k} = {v}" for k, v in data.get("internal_metrics", {}).items()])

# Parse Query Plans (average repeated operators per plan to save tokens)
raw_plans = data.get("query_plans", [])
processed_plans = []

for plan in raw_plans:
    matches = re.findall(r'([A-Za-z0-9_ ]+)\(cost=([0-9.]+)\)', plan)
    op_costs = defaultdict(list)
    for op, cost in matches:
        op_costs[op.strip()].append(float(cost))
        
    avg_ops = []
    for op, costs in op_costs.items():
        avg_cost = sum(costs) / len(costs)
        avg_ops.append(f"{op}(cost={avg_cost:.1f})")
        
    processed_plans.append(" ".join(avg_ops))

unique_plans = list(dict.fromkeys(processed_plans))
q_plan_summary = " ".join(unique_plans)
if len(unique_plans) < len(raw_plans):
    print(f"ℹ️ Deduplicated {len(raw_plans) - len(unique_plans)} repeating query plan(s) to save tokens.")

# Parse Workload Features (extract top-level stats and operator proportions)
wf = data.get("workload_features", {})
wf_flat = [f"{k} = {v}" for k, v in wf.items() if not isinstance(v, dict) and v is not None]
op_props = wf.get("operator_proportions", {})
wf_flat.extend([f"{k} = {v}" for k, v in op_props.items() if v is not None])
workload_features_str = ", ".join(wf_flat)

# Prepare the Instruction
instruction = (
    f"You are an expert {db_name} Database Administrator tuning a server running on {hw_specs} hardware. "
    "Provide the optimal discrete bucket configurations for the following metrics as a single JSON object.\n\n"
    f"WORKLOAD FEATURES: {workload_features_str}\n"
    f"INTERNAL SYSTEM METRICS: {metrics_str}\n"
    f"QUERY PLANS: {q_plan_summary}\n\n"
    "JSON Configuration:"
)

# Format with the 'Pre-fill' { to anchor the response
prompt = f"[INST] {instruction} [/INST] {{"

print("PROMPT:")
print("-" * 50)
print(prompt)
print("-" * 50)

# Push to GPU
inputs = tokenizer(prompt, return_tensors="pt", add_special_tokens=True).to("cuda")

# 3. Generate Sequential Configurations

In [ ]:
output_path = "/kaggle/working/base_model_job_configs.json"
all_configs = []

print("Generating 8 different configurations sequentially to save GPU memory...")

for i in range(8):
    # Generate one configuration at a time
    with torch.no_grad():
        outputs = model.generate(
            **inputs, 
            max_new_tokens=1024, 
            temperature=0.7, # Slightly lower for more stable JSON
            do_sample=True,
            repetition_penalty=1.1,
            eos_token_id=tokenizer.eos_token_id
        )

    # Decode and manually add back the anchored '{'
    generated_text = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    full_json = "{\"" + generated_text
    
    print(f"GENERATED CONFIGURATION {i+1}:")
    print("-" * 50)
    print(full_json)
    print("-" * 50)

    try:
        # Try parsing the output to ensure it's valid JSON before saving.
        parsed_json = json.loads(full_json)
        all_configs.append(parsed_json)
    except json.JSONDecodeError as e:
        print(f"\n⚠️ Warning: Configuration {i+1} was not perfectly formatted JSON ({e}).")
        all_configs.append({"raw_text": full_json})

# Save all 8 to JSON File
with open(output_path, "w") as f:
    json.dump(all_configs, f, indent=4)
    
print(f"\n✅ Successfully saved 8 BASE MODEL configurations to: {output_path}")